# Projet 12 - Système de prédiction de rendement d'une culture spécifique et de recommandation de la culture la plus adapté au caractéristique de la parcelle.


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path

#### Instanciation des dataframes

In [2]:
ROOT = Path.cwd().parent
DATA_FILE = ROOT/"data"

crop_yield = pd.read_csv(DATA_FILE/"crop_yield.csv")
pesticides = pd.read_csv(DATA_FILE/"pesticides.csv")
rainfall = pd.read_csv(DATA_FILE/"rainfall.csv")
temp = pd.read_csv(DATA_FILE/"temp.csv")
yield_df = pd.read_csv(DATA_FILE/"yield_df.csv")
yield2 = pd.read_csv(DATA_FILE/"yield.csv")

#### Data Exploration

In [3]:
#Lecture de crop_yield.csv 
display(crop_yield)



,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251
...,...,...,...,...,...,...,...,...,...,...
999995,West,Silt,Rice,302.805345,27.987428,False,False,Sunny,76,1.347586
999996,South,Chalky,Barley,932.991383,39.661039,True,False,Rainy,93,7.311594
999997,North,Peaty,Cotton,867.362046,24.370042,True,False,Cloudy,108,5.763182
999998,West,Silt,Wheat,492.812857,33.045505,False,False,Sunny,102,2.070159


10 VARIABLES :

Region : Orientation de la parcelle (STRING)
Soil_Type : Le type de sol de l'exploitation (STRING)
Crop : Type de culture (STRING)
Rainfall_mm : milimètres de pluie qui tombés sur l'exploitation (sur un an ?) (FLOAT)
Temperatur_Celsius : Température moyenne (FLOAT)
Fertilizer_Used : Utilisation de fertilisant (BOOLEEN)
Irrigation_Used : Irrigation (BOOLEEN)
Weather_Condition : Météo (STRING)
Days_to_Harvest : Nb de jour avant la récolte (INTEGER)
Yield_tons_per_hectare : Rendement en tonne par hectare (FLOAT)

Identification des cibles : 
- Prédiction du rendement : Yield_tons_per_hectare


Une ligne = une parcelle agricole fictive

In [4]:
#Lecture de yield_df.csv 
display(yield_df)


,Unnamed: 0,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,0,Albania,Maize,1990,36613,1485.0,121.00,16.37
1,1,Albania,Potatoes,1990,66667,1485.0,121.00,16.37
2,2,Albania,"Rice, paddy",1990,23333,1485.0,121.00,16.37
3,3,Albania,Sorghum,1990,12500,1485.0,121.00,16.37
4,4,Albania,Soybeans,1990,7000,1485.0,121.00,16.37
...,...,...,...,...,...,...,...,...
28237,28237,Zimbabwe,"Rice, paddy",2013,22581,657.0,2550.07,19.76
28238,28238,Zimbabwe,Sorghum,2013,3066,657.0,2550.07,19.76
28239,28239,Zimbabwe,Soybeans,2013,13142,657.0,2550.07,19.76
28240,28240,Zimbabwe,Sweet potatoes,2013,22222,657.0,2550.07,19.76


8 variables : 
Unnamed : 0 : ?? (INTEGER)
Area : Pays (STRING)
Item : Type d'élément planté (STRING)
Year : Année (INTEGER)
hg/ha_yield : hectogramme par hectare (INTEGER)
average_rain_fall_mm_per_year : mm de pluie par année (FLOAT)
pesticides_tonnes : tonnes de pesticide (FLOAT)
avg_temp : température moyenne (FLOAT)

Yield_df est composé du merge de pesticides, rainfall, temp, yield2. Mais il ne conserve pas tous les éléments, il nous faudra donc refaire une fusion pour s'assurer des datas.

In [5]:
#Lecture de pesticides.csv 
display(pesticides)


,Domain,Area,Element,Item,Year,Unit,Value
0,Pesticides Use,Albania,Use,Pesticides (total),1990,tonnes of active ingredients,121.00
1,Pesticides Use,Albania,Use,Pesticides (total),1991,tonnes of active ingredients,121.00
2,Pesticides Use,Albania,Use,Pesticides (total),1992,tonnes of active ingredients,121.00
3,Pesticides Use,Albania,Use,Pesticides (total),1993,tonnes of active ingredients,121.00
4,Pesticides Use,Albania,Use,Pesticides (total),1994,tonnes of active ingredients,201.00
...,...,...,...,...,...,...,...
4344,Pesticides Use,Zimbabwe,Use,Pesticides (total),2012,tonnes of active ingredients,3375.53
4345,Pesticides Use,Zimbabwe,Use,Pesticides (total),2013,tonnes of active ingredients,2550.07
4346,Pesticides Use,Zimbabwe,Use,Pesticides (total),2014,tonnes of active ingredients,2185.07
4347,Pesticides Use,Zimbabwe,Use,Pesticides (total),2015,tonnes of active ingredients,2185.07


7 variables :

Domain : Utilisation de pesticides (STRING)
Area : Pays (STRING)
Element : Utilisation d'élément (STRING)
Item : Le ou les pesticides utilisés (STRING)
Year : Année (INT)
Unit : Unité utilisée (STRING)
Value : quantité utilisé (FLOAT)

In [6]:
#Lecture de rainfall.csv 
display(rainfall)


,Area,Year,average_rain_fall_mm_per_year
0,Afghanistan,1985,327
1,Afghanistan,1986,327
2,Afghanistan,1987,327
3,Afghanistan,1989,327
4,Afghanistan,1990,327
...,...,...,...
6722,Zimbabwe,2013,657
6723,Zimbabwe,2014,657
6724,Zimbabwe,2015,657
6725,Zimbabwe,2016,657


3 variables :
Area : Pays (STRING)
Year : Année (INTEGER)
average_rain_fall_mm_per_year : mm de pluie tombée par an (INTEGER)

In [7]:
#Lecture de temp.csv 
display(temp)


,year,country,avg_temp
0,1849,Côte D'Ivoire,25.58
1,1850,Côte D'Ivoire,25.52
2,1851,Côte D'Ivoire,25.67
3,1852,Côte D'Ivoire,NaN
4,1853,Côte D'Ivoire,NaN
...,...,...,...
71306,2009,Mexico,21.76
71307,2010,Mexico,20.90
71308,2011,Mexico,21.55
71309,2012,Mexico,21.52


3 variables : 
year : année (INTEGER)
country : pays (STRING)
avg_temp : temperature moyenne (FLOAT)

In [8]:
#Lecture de yield2.csv 
display(yield2)


,Domain Code,Domain,Area Code,Area,Element Code,Element,Item Code,Item,Year Code,Year,Unit,Value
0,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1961,1961,hg/ha,14000
1,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1962,1962,hg/ha,14000
2,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1963,1963,hg/ha,14260
3,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1964,1964,hg/ha,14257
4,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1965,1965,hg/ha,14400
...,...,...,...,...,...,...,...,...,...,...,...,...
56712,QC,Crops,181,Zimbabwe,5419,Yield,15,Wheat,2012,2012,hg/ha,24420
56713,QC,Crops,181,Zimbabwe,5419,Yield,15,Wheat,2013,2013,hg/ha,22888
56714,QC,Crops,181,Zimbabwe,5419,Yield,15,Wheat,2014,2014,hg/ha,21357
56715,QC,Crops,181,Zimbabwe,5419,Yield,15,Wheat,2015,2015,hg/ha,19826


12 variables : 
Domain Code : 
Domain : 
Area Code : Code de l'endroit (INTEGER)
Area :  Pays (STRING)
Element code : Numéro du pesticide 
Element : Culture
Item Code : Identifiant de l'item (INTEGER)
item : Nom de l'item (STRING)
Year Code : Année (INTEGER)
Year : Année (INTEGER) - identique à Year Code (supprimer Year code)
Unit : Unité (STRING)
Value : Surface de la parcelle ??(INTEGER)

In [9]:
#Informations de bases sur les dfs
liste_df = [crop_yield, yield_df, pesticides, rainfall, temp, yield2]
for df in liste_df:

    print("-"*200)
    print("DF shapes :")
    print(df.shape)
    print("-"*100)
    print("Value count is :")
    print(df.value_counts().sum())
    print("-"*100)
    print("Number of NaN values")
    print(df.isnull().sum())
    print("-"*100)
    print("Number of duplicated")
    print(df.duplicated().sum())
    print("-"*100)
    print("Stats de base :")
    print(df.describe())
    



--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
DF shapes :
(1000000, 10)
----------------------------------------------------------------------------------------------------
Value count is :
1000000
----------------------------------------------------------------------------------------------------
Number of NaN values
Region                    0
Soil_Type                 0
Crop                      0
Rainfall_mm               0
Temperature_Celsius       0
Fertilizer_Used           0
Irrigation_Used           0
Weather_Condition         0
Days_to_Harvest           0
Yield_tons_per_hectare    0
dtype: int64
----------------------------------------------------------------------------------------------------
Number of duplicated
0
----------------------------------------------------------------------------------------------------
Stats d

In [10]:
#Exploration précise par df 
#DF crop_yield
for col in crop_yield.columns:
    print(crop_yield[col].value_counts(dropna=False).head(20))
    print("-" * 50)

print("-"*200)

for col in pesticides.columns:
    print(pesticides[col].value_counts(dropna=False))
    print("-"*50)

print("-"*200)

for col in rainfall.columns:
    print(rainfall[col].value_counts(dropna=False))
    print("-"*50)

print("-"*200)

for col in temp.columns:
    print(temp[col].value_counts(dropna=False))
    print("-"*50)

print("-"*200)

for col in yield_df.columns:
    print(yield_df[col].value_counts(dropna=False))
    print("-"*50)

print("-"*200)

for col in yield2.columns:
    print(yield2[col].value_counts(dropna=False))
    print("-"*50)

print("-"*200)

Region
North    250173
West     250074
South    250054
East     249699
Name: count, dtype: int64
--------------------------------------------------
Soil_Type
Sandy     167119
Loam      166795
Chalky    166779
Silt      166672
Clay      166352
Peaty     166283
Name: count, dtype: int64
--------------------------------------------------
Crop
Maize      166824
Rice       166792
Barley     166777
Wheat      166673
Cotton     166585
Soybean    166349
Name: count, dtype: int64
--------------------------------------------------
Rainfall_mm
897.077239    1
992.673282    1
147.998025    1
986.866331    1
730.379174    1
797.471182    1
357.902357    1
441.131154    1
181.587861    1
395.048968    1
385.135314    1
145.300681    1
607.150252    1
929.123735    1
621.778388    1
874.456744    1
787.084306    1
416.898632    1
977.259083    1
888.207630    1
Name: count, dtype: int64
--------------------------------------------------
Temperature_Celsius
27.676966    1
18.026142    1
29.794042    1

temp : 6 958 doublons et jusqu’à 52 lignes par (pays, année), parce que les données viennent de villes ;
yield_df : 28 242 lignes pour seulement 13 130 combinaisons uniques, à cause de ce problème de temp, avec seulement 101 pays. 
rainfall : des valeurs "..", des NaN, un espace dans le nom de colonne, et une pluie constante par pays ;
les noms de pays différents entre les sources (FAO, Banque mondiale…).

#### Data Cleaning 


In [11]:
# Nettoyage des CSV nécessaires à la construction de yield_df:

# ---------- yield2 ----------
yield2_clean = yield2[["Area", "Item", "Year", "Value"]].copy()  # Colonnes utiles : les autres sont constantes ou  codes redondants
yield2_clean["Area"] = yield2_clean["Area"].str.strip()
yield2_clean["Value"] = yield2_clean["Value"] / 10000  # hg/ha -> t/ha
yield2_clean = yield2_clean.rename(columns={"Value": "yield_t_ha"})
# Suppression de l'agrégat "China" (= Chine continentale + Taïwan + Hong Kong, déjà présents séparément), évite  double comptage
yield2_clean = yield2_clean.loc[yield2_clean["Area"] != "China"]

# ---------- pesticides ----------
pesticides_clean = pesticides[["Area", "Year", "Value"]].copy()
pesticides_clean["Area"] = pesticides_clean["Area"].str.strip()
pesticides_clean = pesticides_clean.rename(columns={"Value": "pesticides_tonnes"})

# ---------- Correspondance des noms de pays vers la référence FAO ----------
map_rainfall = {
    "United States": "United States of America",
    "Russia": "Russian Federation",
    "China": "China, mainland",
    "Hong Kong SAR, China": "China, Hong Kong SAR",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Congo, Dem. Rep.": "Democratic Republic of the Congo",
    "Congo, Rep.": "Congo",
    "Bolivia": "Bolivia (Plurinational State of)",
    "Iran": "Iran (Islamic Republic of)",
    "Venezuela, RB": "Venezuela (Bolivarian Republic of)",
    "South Korea": "Republic of Korea",
    "North Korea": "Democratic People's Republic of Korea",
    "Lao PDR": "Lao People's Democratic Republic",
    "Vietnam": "Viet Nam",
    "Syria": "Syrian Arab Republic",
    "Tanzania": "United Republic of Tanzania",
    "Moldova": "Republic of Moldova",
    "Macedonia": "The former Yugoslav Republic of Macedonia",
    "Czech Republic": "Czechia",
    "Slovak Republic": "Slovakia",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Micronesia": "Micronesia (Federated States of)",
    "St. Kitts and Nevis": "Saint Kitts and Nevis",
    "St. Lucia": "Saint Lucia",
    "St. Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "West Bank and Gaza": "Occupied Palestinian Territory",
}

map_temp = {
    "United States": "United States of America",
    "Russia": "Russian Federation",
    "China": "China, mainland",
    "Taiwan": "China, Taiwan Province of",
    "Hong Kong": "China, Hong Kong SAR",
    "Côte D'Ivoire": "Côte d'Ivoire",
    "Congo (Democratic Republic Of The)": "Democratic Republic of the Congo",
    "Bolivia": "Bolivia (Plurinational State of)",
    "Iran": "Iran (Islamic Republic of)",
    "Venezuela": "Venezuela (Bolivarian Republic of)",
    "South Korea": "Republic of Korea",
    "Laos": "Lao People's Democratic Republic",
    "Vietnam": "Viet Nam",
    "Syria": "Syrian Arab Republic",
    "Tanzania": "United Republic of Tanzania",
    "Moldova": "Republic of Moldova",
    "Macedonia": "The former Yugoslav Republic of Macedonia",
    "Czech Republic": "Czechia",
    "Bosnia And Herzegovina": "Bosnia and Herzegovina",
    "Guinea Bissau": "Guinea-Bissau",
}

# ---------- rainfall ----------
rainfall_clean = rainfall.copy()
rainfall_clean.columns = rainfall_clean.columns.str.strip()  # " Area" -> "Area"
rainfall_clean["Area"] = rainfall_clean["Area"].str.strip().replace(map_rainfall)
rainfall_clean["average_rain_fall_mm_per_year"] = pd.to_numeric(
    rainfall_clean["average_rain_fall_mm_per_year"], errors="coerce"  # ".." -> NaN
)
# Pluie quasi constante par pays (moyenne climatique) -> une valeur par pays
rainfall_clean = (
    rainfall_clean.groupby("Area", as_index=False)["average_rain_fall_mm_per_year"].mean()
)
rainfall_clean = rainfall_clean.dropna(subset=["average_rain_fall_mm_per_year"])  # Pays sans aucune donnée

# ---------- temp ----------
temp_clean = temp.rename(columns={"country": "Area", "year": "Year"})
temp_clean = temp_clean.loc[temp_clean["Year"] >= 1990].copy()  # Période commune avec pesticides
temp_clean["Area"] = temp_clean["Area"].str.strip().replace(map_temp)
# Plusieurs relevés (villes) par pays et par année -> moyenne
temp_clean = temp_clean.groupby(["Area", "Year"], as_index=False)["avg_temp"].mean()


print(yield2_clean.shape)
print(pesticides_clean.shape)
print(rainfall_clean.shape)
print(temp_clean.shape)

(56269, 4)
(4349, 3)
(192, 2)
(3288, 3)


In [ ]:
#Fusion des 4 dfs nettoyés pour recréer un df propre et complet qui servira à la fusion finale avec le crop_yield
#Yield2_clean + pesticides_clean
yield_merged = yield2_clean.merge(
    pesticides_clean,
    on=["Area", "Year"],
    how= "inner",
)
print(yield_merged.shape)

#Yield_merged + rainfall_clean
yield_merged = yield_merged.merge(
    rainfall_clean,
    on=["Area"],
    how= "inner",
)
print(yield_merged.shape)


#Yield_merged + temp_clean
yield_merged = yield_merged.merge(
    temp_clean,
    on=["Area", "Year"],
    how= "inner",
)
print(yield_merged.shape)

,Area,Item,Year,yield_t_ha,pesticides_tonnes
0,Albania,Maize,1990,3.6613,121.00
1,Albania,Maize,1991,2.9068,121.00
2,Albania,Maize,1992,2.4876,121.00
3,Albania,Maize,1993,2.4185,121.00
4,Albania,Maize,1994,2.5848,201.00
...,...,...,...,...,...
24148,Zimbabwe,Wheat,2012,2.4420,3375.53
24149,Zimbabwe,Wheat,2013,2.2888,2550.07
24150,Zimbabwe,Wheat,2014,2.1357,2185.07
24151,Zimbabwe,Wheat,2015,1.9826,2185.07


,Area,Item,Year,yield_t_ha,pesticides_tonnes,average_rain_fall_mm_per_year
0,Albania,Maize,1990,3.6613,121.00,1485.0
1,Albania,Maize,1991,2.9068,121.00,1485.0
2,Albania,Maize,1992,2.4876,121.00,1485.0
3,Albania,Maize,1993,2.4185,121.00,1485.0
4,Albania,Maize,1994,2.5848,201.00,1485.0
...,...,...,...,...,...,...
23115,Zimbabwe,Wheat,2012,2.4420,3375.53,657.0
23116,Zimbabwe,Wheat,2013,2.2888,2550.07,657.0
23117,Zimbabwe,Wheat,2014,2.1357,2185.07,657.0
23118,Zimbabwe,Wheat,2015,1.9826,2185.07,657.0


,Area,Item,Year,yield_t_ha,pesticides_tonnes,average_rain_fall_mm_per_year,avg_temp
0,Albania,Maize,1990,3.6613,121.00,1485.0,16.37
1,Albania,Maize,1991,2.9068,121.00,1485.0,15.36
2,Albania,Maize,1992,2.4876,121.00,1485.0,16.06
3,Albania,Maize,1993,2.4185,121.00,1485.0,16.05
4,Albania,Maize,1994,2.5848,201.00,1485.0,16.96
...,...,...,...,...,...,...,...
16487,Zimbabwe,Wheat,2009,3.0000,3269.99,657.0,20.52
16488,Zimbabwe,Wheat,2010,2.7681,3305.17,657.0,21.17
16489,Zimbabwe,Wheat,2011,2.6274,3340.35,657.0,20.78
16490,Zimbabwe,Wheat,2012,2.4420,3375.53,657.0,20.52


Le merge a été réalisé sur les pays et les années pour conservé un maximum de données. Pour rainfall_clean, le merge a été réalisé seulement sur les pays car nous avons récupéré la moyenne par pays des précipitations. De plus, la jointure a été fait sur inner pour ne pas créer de ligne avec NaN a géré plus tard. De plus, on ne récupère que les infos des années de 1990 et au delà, les années précédentes présentant beaucoup de NaN.

In [13]:
#Remplacement des types de culture de yield_merged en ceux de crop :
# Vérification des cultures
print(yield_merged["Item"].unique())
print(crop_yield["Crop"].unique())

# Harmonisation des noms de cultures
yield_merged["Item"] = yield_merged["Item"].replace({
    "Rice, paddy": "Rice",
    "Soybeans": "Soybean"
})

# Renommer la colonne
yield_merged = yield_merged.rename(
    columns={"Item": "Crop"}
)

# Vérification
print(yield_merged.columns.tolist())
print(yield_merged["Crop"].unique())


<StringArray>
[               'Maize',             'Potatoes',          'Rice, paddy',
              'Sorghum',             'Soybeans',                'Wheat',
              'Cassava',       'Sweet potatoes', 'Plantains and others',
                 'Yams']
Length: 10, dtype: str
<StringArray>
['Cotton', 'Rice', 'Barley', 'Soybean', 'Wheat', 'Maize']
Length: 6, dtype: str
['Area', 'Crop', 'Year', 'yield_t_ha', 'pesticides_tonnes', 'average_rain_fall_mm_per_year', 'avg_temp']
<StringArray>
[               'Maize',             'Potatoes',                 'Rice',
              'Sorghum',              'Soybean',                'Wheat',
              'Cassava',       'Sweet potatoes', 'Plantains and others',
                 'Yams']
Length: 10, dtype: str


Yield_merged est clean. Nous allons maintenant nettoyer crop_yield avant le merge entre ces deux dfs.

In [14]:
#Nettoyage de crop_yield:
crop_yield_clean = crop_yield.loc[crop_yield["Yield_tons_per_hectare"]>0]
crop_yield_clean = crop_yield_clean[""]

KeyError: ''